# C09 · Train the court-keypoint detector (Google Colab)

**hoop-reid-tracker**, court chapter C09. This notebook trains YOLO26-pose to find the **33 court keypoints** (court corners, lane corners, free-throw line ends, 3-pt corners, centre line, ...) in any broadcast frame. Each keypoint has a name, so a frame's detections give point pairs (image ↔ court in metres), and from them a **homography** for that frame, with no hand anchors.

### Why on Colab
On the laptop (CPU) a 3 h budget gave **2 epochs** (pose mAP50-95 0.075): far too few. A T4 does an epoch in a few minutes.

### What the model learns
* One object, the **court**, with 33 keypoints. A keypoint is *visible* (2) whenever it projects inside the image, even when a player or the score bar hides it: its position is known exactly from the homography, and the model must learn to **infer** it from the lines around it.
* The hoops (09, 33) are in the air and are never labelled.

### Data (all labels come from a court homography)
| source | what | split |
|---|---|---|
| `v22` | 850 NBA 2025 broadcast frames (C01/C02) | train / val / test |
| `deepsport` | DeepSportRadar, 15 French LNB arenas (FIBA), calibrated cameras | train / val / test **by arena** |
| `sportsmot` | SportsMOT **train** clips, frames the C07 clip camera accepted | train only |
| `synth` | 2000 domain-randomised courts (NBA / FIBA / NCAA) | train only |

Validation and test are **real images only**. The SportsMOT **validation** clips are never here: they are the hand-annotated test set (`test_set.json`), measured on the laptop after training.

### Before running (on your laptop)
1. `python scripts/court/c09_pack_colab.py` creates `data/court_kp.zip` (~750 MB).
2. Upload `court_kp.zip` to Google Drive, in `MyDrive/hooptrack/`.
3. In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

Weights are written to Drive every epoch. If Colab disconnects, run sections 1–3 again and then section 5: training **resumes**.

## 1 · GPU and libraries
`nvidia-smi` must show a GPU. If it says *command not found*, change the runtime type.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "ultralytics>=8.4"   # YOLO26 needs ultralytics 8.4 or newer
import ultralytics
ultralytics.checks()

## 2 · Configuration
* `MODEL`: `yolo26n-pose`, the smallest pose model, COCO-pretrained. Small on purpose: it will run next to the tracker (and later on a Jetson). Try `yolo26s-pose.pt` if `n` underfits.
* `IMGSZ = 640`: court lines stay visible; the laptop pipeline must use the same size (`--imgsz 640`).
* `EPOCHS`, `PATIENCE`: up to 150 epochs, early stop after 30 without improvement on val.

In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/hooptrack")   # where you uploaded court_kp.zip
    WORK = Path("/content")                                # fast local disk of the Colab machine
else:
    DRIVE_DIR = Path(os.environ.get("HOOPTRACK_DIR", "data"))
    WORK = Path(os.environ.get("HOOPTRACK_WORK", "/tmp/hooptrack"))

ZIP = DRIVE_DIR / "court_kp.zip"
RUNS = DRIVE_DIR / "runs"            # checkpoints on Drive: they survive a disconnection
NAME = "c09_yolo26n_pose_640"

MODEL = os.environ.get("HOOPTRACK_MODEL", "yolo26n-pose.pt")
IMGSZ = int(os.environ.get("HOOPTRACK_IMGSZ", 640))
EPOCHS = int(os.environ.get("HOOPTRACK_EPOCHS", 150))
PATIENCE = 30
BATCH = int(os.environ.get("HOOPTRACK_BATCH", 32))   # yolo26n-pose @640 fits easily on a T4
HOURS = None                         # e.g. 3.0 to fit a session; None = no time limit
WORKERS = 2                          # Colab has 2 CPU cores

assert ZIP.is_file(), f"{ZIP} not found: upload court_kp.zip to MyDrive/hooptrack/"
RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print("zip:", ZIP, f"({ZIP.stat().st_size / 1e6:.0f} MB) | runs:", RUNS)

## 3 · Unpack the dataset on the local disk
Reading thousands of small files from Drive is very slow, so the zip is copied and unpacked once (~1–2 min). `data.yaml` lists v22 as `../court_keypoints/yolo_v22`, which is why the zip holds both folders side by side.

In [ ]:
import shutil, zipfile, yaml

DATA_ROOT = WORK / "court_kp"
if not (DATA_ROOT / "data.yaml").is_file():
    local_zip = WORK / "court_kp.zip"
    shutil.copy(ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(WORK)
    local_zip.unlink()

cfg = yaml.safe_load((DATA_ROOT / "data.yaml").read_text())
cfg["path"] = str(DATA_ROOT)
DATA = DATA_ROOT / "data_colab.yaml"
DATA.write_text(yaml.safe_dump(cfg, sort_keys=False))
KPT_NAMES = cfg["kpt_names"][0] if isinstance(cfg["kpt_names"], dict) else cfg["kpt_names"]

for split in ("train", "val", "test"):
    for d in cfg[split]:
        n = len(list((DATA_ROOT / d).glob("*.jpg")))
        print(f"{split:5s} {d:45s} {n:5d} images")

### Look at the labels
Always check labels before training. Red dots are the keypoints the model must find, with their names. Points hidden by players are labelled too (the model has to infer them).

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np

def draw_labels(img_path: Path):
    img = cv2.imread(str(img_path)); h, w = img.shape[:2]
    lbl = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")
    vals = lbl.read_text().split() if lbl.is_file() else []
    if vals:
        k = np.array(vals[5:], float).reshape(-1, 3)
        for name, (x, y, v) in zip(KPT_NAMES, k):
            if v > 0:
                c = (int(x * w), int(y * h))
                cv2.circle(img, c, 6, (0, 0, 255), -1)
                cv2.putText(img, str(name), (c[0] + 6, c[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return img

random.seed(0)
samples = []
for d in cfg["train"]:                       # one image per source
    imgs = sorted((DATA_ROOT / d).glob("*.jpg"))
    if imgs:
        samples.append(random.choice(imgs))
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, p in zip(axes.ravel(), samples):
    ax.imshow(draw_labels(p)[..., ::-1]); ax.set_title(p.parent.parent.parent.name + " | " + p.stem, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 4 · Training
Same augmentation choices as the laptop script (`scripts/court/train_keypoints.py`), and why:

| setting | value | why |
|---|---|---|
| `mosaic` | 0 | mosaic glues 4 images: it would cut the court, the single object we predict, into pieces that never occur |
| `fliplr` + `flip_idx` | 0.5 | a mirrored frame is a valid frame of the other half; `flip_idx` (in data.yaml) swaps left/right keypoint names |
| `flipud`, `degrees` | 0 | courts are never upside down; broadcast cameras do not roll |
| `scale`, `translate` | 0.3, 0.1 | zoom and pan |
| `hsv_h / hsv_s / hsv_v` | 0.1 / 0.6 / 0.4 | floor and paint colours vary a lot between arenas |

No backbone freezing here: on a GPU the whole network can adapt to lines instead of people.

**If Colab disconnects:** run sections 1–3 again and then this cell. It finds `last.pt` on Drive and resumes.

In [ ]:
import torch
from ultralytics import YOLO

last = RUNS / NAME / "weights" / "last.pt"
# A finished run stores epoch = -1 in last.pt (optimizer stripped): nothing to resume.
finished = last.is_file() and torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) == -1
if finished:
    print("Training already finished:", last.parent, "(delete or rename the run folder to train again)")
elif last.is_file():
    print("Resuming from", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(MODEL).train(
        data=str(DATA), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, workers=WORKERS,
        time=HOURS, project=str(RUNS), name=NAME, exist_ok=True, seed=0, plots=True, cache="ram",
        mosaic=0.0, mixup=0.0, fliplr=0.5, flipud=0.0, degrees=0.0, translate=0.1, scale=0.3,
        hsv_h=0.1, hsv_s=0.6, hsv_v=0.4,
    )
BEST = RUNS / NAME / "weights" / "best.pt"
print("best weights:", BEST)

### Training curves
Losses should go down; val metrics (especially **pose mAP50-95**) should go up and flatten. The laptop run ended at pose mAP50-95 **0.075** after 2 epochs: that is the number to beat by far.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RUNS / NAME / "results.png"), width=1100))

## 5 · Evaluation on held-out images
* **val**: v22 + DeepSport val arenas (used to choose the best epoch).
* **test**: v22 + DeepSport **test arenas**, never seen.

Pose mAP uses OKS (how close each predicted keypoint is to the true one, relative to the object size). For us the useful number comes later, on the laptop: **metres of error of the resulting homography** on the hand-annotated SportsMOT test clips.

In [ ]:
import json
import pandas as pd

rows = {}
model = YOLO(str(BEST))
for split in ("val", "test"):
    m = model.val(data=str(DATA), split=split, imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False,
                  project=str(WORK / "val_runs"), name=split, exist_ok=True)
    rows[split] = {"box mAP50": round(m.box.map50, 3), "pose P": round(m.pose.mp, 3), "pose R": round(m.pose.mr, 3),
                   "pose mAP50": round(m.pose.map50, 3), "pose mAP50-95": round(m.pose.map, 3)}
table = pd.DataFrame(rows).T
display(table)
(RUNS / NAME / "eval_c09.json").write_text(json.dumps({"model": MODEL, "imgsz": IMGSZ, "metrics": rows}, indent=2))

### Predictions on test images
Red = predicted keypoints with confidence ≥ 0.5; green = labels. A good model puts red on green even where players hide the point.

In [ ]:
test_imgs = [p for d in cfg["test"] for p in sorted((DATA_ROOT / d).glob("*.jpg"))]
random.seed(1)
show = random.sample(test_imgs, min(4, len(test_imgs)))
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, p in zip(axes.ravel(), show):
    img = cv2.imread(str(p)); h, w = img.shape[:2]
    lbl = Path(str(p).replace("/images/", "/labels/")).with_suffix(".txt").read_text().split()
    if lbl:
        for x, y, v in np.array(lbl[5:], float).reshape(-1, 3):
            if v > 0:
                cv2.circle(img, (int(x * w), int(y * h)), 7, (0, 255, 0), 2)
    r = model.predict(str(p), imgsz=IMGSZ, verbose=False)[0]
    if r.keypoints is not None and len(r.keypoints):
        xy, cf = r.keypoints.xy[0].cpu().numpy(), r.keypoints.conf[0].cpu().numpy()
        for (x, y), c in zip(xy, cf):
            if c >= 0.5:
                cv2.circle(img, (int(x), int(y)), 4, (0, 0, 255), -1)
    ax.imshow(img[..., ::-1]); ax.set_title(p.stem, fontsize=9); ax.axis("off")
plt.tight_layout()
plt.savefig(RUNS / NAME / "test_predictions.jpg", dpi=80)
plt.show()
print("Download:", BEST)

## 6 · Take it home and measure the homographies
1. Download `MyDrive/hooptrack/runs/c09_yolo26n_pose_640/weights/best.pt` and save it in the repo as `outputs/court_runs/c09_colab/weights/best.pt`.
2. On the laptop (repo root, venv active): homographies on the test clips, clip-camera verification (C07), and the error against the hand-annotated test set:

```powershell
python scripts/court/c09_detect_homographies.py --weights outputs/court_runs/c09_colab/weights/best.pt --imgsz 640 `
    --sequence v_00HRwkvvjtQ_c001 v_00HRwkvvjtQ_c005 v_5ekaksddqrc_c003 v_BgwzTUxJaeU_c008 v_BgwzTUxJaeU_c012 v_4r8QL_wglzQ_c001 `
    --out outputs/court_c09
python scripts/court/verify_clip_camera.py --in outputs/court_c09 `
    --sequence v_00HRwkvvjtQ_c001 v_00HRwkvvjtQ_c005 v_5ekaksddqrc_c003 v_BgwzTUxJaeU_c008 v_BgwzTUxJaeU_c012 v_4r8QL_wglzQ_c001
python scripts/court/eval_court_test.py --method c05=outputs/court_auto_edges c05+c07=outputs/court_verified/auto_edges `
    c09=outputs/court_c09 c09+c07=outputs/court_verified/c09 `
    --out experiments/court/C09_learned_keypoints/eval_test.json
```

The number that matters: **c09+c07** with more frames covered than c05+c07 and **0 wrong** courts (> 1 m).